# 03 Baseline and time split
- **Train:** 2022–23. **Validate:** 2024. **Test:** 2025, locked until notebook 05.
- **Baseline:** the top 3 on the grid make the podium.
- **Metrics:** podium hits per race (0–3), plus Brier score.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

features = pd.read_csv(PROCESSED_DATA_DIR / "features.csv", parse_dates=["EventDate"])
features.shape

(1836, 26)

## Time split
Whole seasons only. Test set is dropped here so nothing can peek at it.

In [2]:
TRAIN_YEARS = [2022, 2023]
VALID_YEARS = [2024]

train = features[features["Year"].isin(TRAIN_YEARS)].copy()
valid = features[features["Year"].isin(VALID_YEARS)].copy()

assert train["EventDate"].max() < valid["EventDate"].min()
for name, df in [("train", train), ("valid", valid)]:
    print(f"{name}: {df['Year'].unique()}, {df.groupby(['Year', 'Round']).ngroups} races, {len(df)} rows")

train: [2022 2023], 44 races, 878 rows
valid: [2024], 24 races, 479 rows


## Metrics
- **Podium hits:** take the top 3 drivers by predicted probability in each race and count how many actually finished on the podium.
- **Brier score:** mean squared error of the probabilities. Lower is better.

In [3]:
def podium_hits(df, prob_col):
    """Hits per race: top 3 by probability vs actual podium (0-3)."""
    top3 = df.sort_values(prob_col, ascending=False).groupby(["Year", "Round"]).head(3)
    return top3.groupby(["Year", "Round"])["Podium"].sum()


def brier(df, prob_col):
    return np.mean((df[prob_col] - df["Podium"]) ** 2)


def evaluate(df, prob_col):
    hits = podium_hits(df, prob_col)
    return {
        "hits_per_race": round(hits.mean(), 3),
        "perfect_races_pct": round((hits == 3).mean() * 100, 1),
        "brier": round(brier(df, prob_col), 4),
    }

## Baseline
The top 3 on the grid are predicted as the podium.

Brier needs probabilities, so each grid slot gets its podium rate **from train only**. Those probabilities rank drivers in grid order, so hits per race equal the pure grid top-3 rule.

In [4]:
grid_rate = train.groupby("GridPosition")["Podium"].mean()
grid_rate

GridPosition
1.0     0.818182
2.0     0.681818
3.0     0.409091
4.0     0.340909
5.0     0.204545
6.0     0.181818
7.0     0.090909
8.0     0.023256
9.0     0.045455
10.0    0.046512
11.0    0.022727
12.0    0.068182
13.0    0.000000
14.0    0.022727
15.0    0.045455
16.0    0.000000
17.0    0.000000
18.0    0.000000
19.0    0.000000
20.0    0.000000
21.0    0.000000
Name: Podium, dtype: float64

In [5]:
def baseline_prob(df):
    # Grid slots never seen in train -> 0
    return df["GridPosition"].map(grid_rate).fillna(0)

train["BaselineProb"] = baseline_prob(train)
valid["BaselineProb"] = baseline_prob(valid)

# Tie-break safety: hits must equal grid top-3 rule exactly
valid["GridTop3"] = (valid["GridPosition"] <= 3).astype(int)
grid_hits = valid[valid["GridTop3"] == 1].groupby(["Year", "Round"])["Podium"].sum()
assert grid_hits.mean() == podium_hits(valid, "BaselineProb").mean()

pd.DataFrame({"train 2022-23": evaluate(train, "BaselineProb"),
              "valid 2024": evaluate(valid, "BaselineProb")})

,train 2022-23,valid 2024
hits_per_race,1.9090,2.0000
perfect_races_pct,13.6000,20.8000
brier,0.0745,0.0699


## Where the baseline misses (2024)

In [6]:
hits = podium_hits(valid, "BaselineProb")
print(hits.value_counts().sort_index())

races = valid.drop_duplicates(["Year", "Round"]).set_index(["Year", "Round"])["GrandPrix"]
worst = hits[hits <= 1].index
races.loc[worst]

Podium
0     1
1     3
2    15
3     5
Name: count, dtype: int64


Year  Round
2024  1          Bahrain Grand Prix
      11        Austrian Grand Prix
      21       São Paulo Grand Prix
      23           Qatar Grand Prix
Name: GrandPrix, dtype: object

In [7]:
# Podium finishers who started outside the top 3: what the model must find
surprises = valid[(valid["Podium"] == 1) & (valid["GridPosition"] > 3)]
print(f"{len(surprises)} of {valid['Podium'].sum()} podiums came from outside the top 3")
surprises[["GrandPrix", "Abbreviation", "Team", "GridPosition", "FinishingPosition"]]

24 of 72 podiums came from outside the top 3


,GrandPrix,Abbreviation,Team,GridPosition,FinishingPosition
881,Bahrain Grand Prix,SAI,Ferrari,4.0,3.0
882,Bahrain Grand Prix,PER,Red Bull,5.0,2.0
921,Australian Grand Prix,LEC,Ferrari,4.0,2.0
940,Japanese Grand Prix,SAI,Ferrari,4.0,3.0
960,Chinese Grand Prix,NOR,McLaren,4.0,2.0
981,Miami Grand Prix,NOR,McLaren,5.0,1.0
1080,Austrian Grand Prix,SAI,Ferrari,4.0,3.0
1083,Austrian Grand Prix,PIA,McLaren,7.0,2.0
1100,British Grand Prix,VER,Red Bull,4.0,2.0
1121,Hungarian Grand Prix,HAM,Mercedes,5.0,3.0


**Number to beat on 2024:** the baseline row above. Notebook 04 models must beat both hits per race and Brier score.